<a href="https://colab.research.google.com/github/BosunL/SEA-VQA/blob/main/Pangea_7B_SEA_CVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Visual Question Answering Example — (Pangea-7B)

This notebook demonstrates Visual Question Answering (VQA) and multilingual/multicultural
Multiple-Choice VQA (SEA-MCQA) using **[neulab/Pangea-7B](https://huggingface.co/neulab/Pangea-7B)**,
a fully open multilingual multimodal LLM covering 39 languages (LLaVA-NeXT architecture with a
Qwen2-7B-Instruct backbone).

This notebook was adapted from a `Salesforce/blip-vqa-base` example. The pipeline (loading a
Google Sheet of images/questions, building MCQA prompts, evaluating English/Teso/Swahili
questions, and visualising results) is unchanged — only the model, processor, prompt format,
and generation call have been swapped out for Pangea-7B.

> **GPU note:** Pangea-7B is an 8B-parameter model. In Colab, go to
> `Runtime > Change runtime type` and select a GPU (an A100 or L4 is recommended for a
> comfortable fp16 load; a T4 will also work but is slower and closer to its 16GB VRAM limit).
> An optional 4-bit quantization cell is included below for memory-constrained GPUs (e.g. T4).

###Install Necessary Libraries

We need a recent version of `transformers` (for `LlavaNextForConditionalGeneration` support),
`accelerate` for device placement, and `Pillow` for image processing. `bitsandbytes` is only
needed if you enable the optional 4-bit quantization cell.

In [ ]:
# Install transformers and other necessary libraries.
# IMPORTANT: do NOT pip-install `torch` on its own here. Colab ships a pre-matched
# torch + torchvision pair; upgrading only `torch` (e.g. `pip install -U torch`) leaves
# the old torchvision binary in place and breaks its compiled ops, causing
# `RuntimeError: operator torchvision::nms does not exist`. So we leave torch/torchvision
# untouched and only install/upgrade the packages we actually need.
!pip install -qqq -U "transformers>=4.45.0" accelerate Pillow sentencepiece
!pip install -qqq pandas datasets matplotlib
# Optional, only needed if you enable 4-bit quantization further below:
!pip install -qqq bitsandbytes

# If you previously hit "RuntimeError: operator torchvision::nms does not exist" (usually
# from an earlier run that upgraded torch on its own), uncomment the two lines below to
# reinstall torch + torchvision as a matched pair, then go to
# Runtime > Restart session before re-running the rest of the notebook.
!pip install -qqq -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
print("Now restart the runtime: Runtime > Restart session, then re-run all cells.")

Now restart the runtime: Runtime > Restart session, then re-run all cells.


### Import Libraries

Import the required modules from `transformers` and `PIL` (Pillow).

In [ ]:
import torch
from PIL import Image
import requests
import io

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Sanity check: catch a broken torch/torchvision pairing here, with a clear message,
# instead of a confusing ModuleNotFoundError later when transformers tries to import
# LlavaNextForConditionalGeneration.
try:
    import torchvision
    from torchvision.ops import nms as _nms_check
    _ = _nms_check(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([0.9]), 0.5)
    print(f"torchvision version: {torchvision.__version__} (compiled ops OK)")
except Exception as e:
    print(
        "torch/torchvision version mismatch detected! "
        "Go to Runtime > Restart session, then re-run all cells without "
        "pip-installing `torch` on its own.\n"
        f"Underlying error: {e}"
    )

Torch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
torchvision version: 0.26.0+cu128 (compiled ops OK)


### Load Pre-trained VQA Model and Processor

We'll use **[neulab/Pangea-7B](https://huggingface.co/neulab/Pangea-7B)**, a multilingual
multimodal LLM. Pangea-7B follows the LLaVA-NeXT architecture, and the model authors provide a
`transformers`-native checkpoint, **[neulab/Pangea-7B-hf](https://huggingface.co/neulab/Pangea-7B-hf)**,
that can be loaded directly with `LlavaNextForConditionalGeneration` (no need to clone the
LLaVA-NeXT repo). This notebook uses that checkpoint — it is the same underlying Pangea-7B
weights, just packaged for `transformers.generate()`.

In [ ]:
# ============================================================
# Load Model and Processor (canonical load — used for the rest of the notebook)
# ============================================================
from transformers import LlavaNextForConditionalGeneration, AutoProcessor

MODEL_ID = "neulab/Pangea-7B-hf"  # transformers-native checkpoint of neulab/Pangea-7B

# Set to True if you're on a memory-constrained GPU (e.g. a free-tier Colab T4).
USE_4BIT_QUANTIZATION = True

print(f"Loading {MODEL_ID} ...")
print("(This is an 8B-parameter multilingual multimodal model — the first download can take a few minutes.)")

processor = AutoProcessor.from_pretrained(MODEL_ID)

if USE_4BIT_QUANTIZATION:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    ).to(0)

# Pangea adds a few special tokens to the Qwen2 tokenizer, so the embedding
# matrix needs to be resized to match (as documented on the model card).
# Skip this when quantized, since resizing a 4-bit-quantized embedding/head is unsupported.
if not USE_4BIT_QUANTIZATION:
    model.resize_token_embeddings(len(processor.tokenizer))

model.eval()
print("Model and processor loaded.\n")

Loading neulab/Pangea-7B-hf ...
(This is an 8B-parameter multilingual multimodal model — the first download can take a few minutes.)


processor_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/735 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Model and processor loaded.





### Model Evaluation

In a real-world scenario, evaluating a VQA model can be done in a few ways.

- **Qualitative Evaluation** (Manual Inspection): For a single example like the one demonstrated above, we can simply look at the image, the question, and the model's answer to see if it makes sense.

- **Quantitative Evaluation** (Using Datasets): This a more rigorous evaluation, you would typically use a VQA benchmark dataset. Such datasets contain thousands or millions of image-question pairs, each with multiple human-annotated answers. The common metrics are:

- **Accuracy**: The percentage of times the model's answer exactly matches one of the ground-truth answers.

- **VQA Score**: A more nuanced metric often used in VQA challenges. For each question, if the model's answer is present in the human answers, it gets a score that's min(human_answers_count / 3, 1). This accounts for cases where multiple correct answers exist.

**To perform quantitative evaluation, you would:**

Load a VQA dataset.
Iterate through the dataset, feeding each image-question pair to your model.
Collect the model's predictions.
Compare the predictions against the ground-truth answers using the chosen metric.


In [ ]:
# Lets try a real world VQA model (Pangea-7B), and MCQA data

import random
import matplotlib.pyplot as plt
from datasets import load_dataset

In [ ]:
# ============================================================
# STEP 1: Model and Processor
# ============================================================
# `model` and `processor` were already loaded above (cell loading neulab/Pangea-7B-hf).
# We simply reuse those objects here, to avoid downloading/loading the 8B-parameter
# model a second time.
print(f"Using model:     {MODEL_ID}")
print(f"Model device:    {model.device}")
print(f"Model dtype:     {model.dtype}")

Using model:     neulab/Pangea-7B-hf
Model device:    cuda:0
Model dtype:     torch.float16


In [ ]:
model.generation_config.pad_token_id = processor.tokenizer.eos_token_id


##Using Google Drive for Direct Image Links

If you prefer to use Google Drive to host your images, follow these steps to get a direct download link:

1.  **Upload your image(s)** to Google Drive.
2.  **Right-click** on the image file and select **'Share'**.
3.  In the sharing dialog, change the access from 'Restricted' to **'Anyone with the link'**.
4.  **Copy the shareable link**. It will look something like this:
    `https://drive.google.com/file/d/FILE_ID/view?usp=sharing`
5.  **Convert the link to a direct download link** by extracting the `FILE_ID` and formatting it as:
    `https://drive.google.com/uc?export=download&id=FILE_ID`

    **Example:**
    If your shareable link is `https://drive.google.com/file/d/1aBcDeFGhIjKlMnOpQrStUvWxYz/view?usp=sharing`,
    your direct download link will be `https://drive.google.com/uc?export=download&id=1aBcDeFGhIjKlMnOpQrStUvWxYz`

6.  **Update your Google Sheet** with these direct download links in your `image_filename` column.

In [ ]:
import os
import pandas as pd
from datasets import Dataset, Features, Value, Image as ImageFeature

# 1. Configuration
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTs9PcGM6qO4MeGzL33BtqTDhFFt7Cq4YPXolFy-lpsdHfP-yBjL_EFvyC2Saoj_AJ9dm9VT9udAJgj/pub?output=csv"
IMAGE_DIR = "/content/images/"

def clean_image_path_or_url(filename):
    """Cleans up whitespace and handles absolute web URLs or local paths."""
    if not isinstance(filename, str):
        return filename
    filename = filename.strip()

    # If it is an internet URL, use it directly. If it is a filename, prepend the folder path.
    if filename.startswith(('http://', 'https://')):
        return filename
    return os.path.join(IMAGE_DIR, filename)

# 2. Extract Data from Google Sheet
print("Fetching Google Sheet...")
df = pd.read_csv(GOOGLE_SHEET_URL)
df.columns = df.columns.str.strip()  # Clean whitespace from headers

# Ensure an Index ID column is present
if 'ID' not in df.columns:
    df['ID'] = range(1, len(df) + 1)

# 3. Map Data Structures for Lazy Loading
if 'image_filename' in df.columns:
    # Set up final resource pointers
    df['image'] = df['image_filename'].apply(clean_image_path_or_url)
    df = df.drop(columns=['image_filename'])

    # Infer metadata schema types for the text columns
    features_dict = {
        col: Value(dtype='string') if dtype == 'object' else
             Value(dtype='int64') if 'int' in str(dtype) else
             Value(dtype='float64') if 'float' in str(dtype) else
             Value(dtype='bool') if 'bool' in str(dtype) else Value(dtype='string')
        for col, dtype in df.dtypes.items() if col != 'image'
    }
    features_dict['image'] = ImageFeature()  # Stream directly from path/URL string on-the-fly

    # Instantiate the stable dataset mapping
    print("Building Hugging Face lazy-loading structure...")
    custom_dataset_from_sheet = Dataset.from_pandas(df, features=Features(features_dict))

    samples = custom_dataset_from_sheet.select(range(len(custom_dataset_from_sheet)))
    print(f"✅ Loaded {len(samples)} samples successfully. System RAM footprint: ~0 MB.")
else:
    print("❌ Error: Column 'image_filename' was not found in your Google Sheet.")

Fetching Google Sheet...
Building Hugging Face lazy-loading structure...
✅ Loaded 51 samples successfully. System RAM footprint: ~0 MB.


After running this cell, `samples` will contain your data loaded from the Google Sheet, with the images processed accordingly. Remember to then uncomment and run cell `302bd8b0` to use this new `custom_dataset_from_sheet` for evaluation.

Now, the `samples` variable contains your custom data, and the subsequent evaluation steps will use this data.

In [ ]:
# To use your custom dataset, comment out the original `load_dataset` cell (JQVDEE5W8ZZR) and uncomment the following line:
dataset = custom_dataset_from_sheet
samples = dataset.select(range(len(custom_dataset_from_sheet)))

In [ ]:
# ============================================================
# STEP 2: Display Sample Images
# ============================================================
print("Displaying sample images...")

# Group samples by a unique identifier for the image.
# For this task, we'll use 'Category' as a grouping key, assuming all samples
# within the same 'Category' are intended to share the same visual context.
grouped_images = {}
for i, sample in enumerate(samples):
    category = sample["Category"] # Use Category for grouping
    if category not in grouped_images:
        grouped_images[category] = {
            "image": sample["image"], # Store the actual image object
            "question_indices": []    # Store original 1-based question numbers
        }
    grouped_images[category]["question_indices"].append(i + 1)

# Determine the number of unique images to plot
num_unique_images = len(grouped_images)
# Adjust figure size dynamically based on the number of unique images
fig, axes = plt.subplots(1, num_unique_images, figsize=(5 * num_unique_images, 4))

# Ensure 'axes' is always an iterable (list) for consistent indexing, even if only one subplot
if num_unique_images == 1:
    axes = [axes]

for idx, (category, group_info) in enumerate(grouped_images.items()):
    image_to_display = group_info["image"]
    question_numbers = group_info["question_indices"]

    # Format the question numbers for the title (e.g., "Q1", "Q1-4")
    q_label = f"Q{question_numbers[0]}"
    if len(question_numbers) > 1:
        q_label += f"-{question_numbers[-1]}"

    # Create the full title including the aggregated question label and category
    full_title = f"{q_label}\n{category}"

    axes[idx].imshow(image_to_display)
    axes[idx].set_title(full_title, fontsize=8)
    axes[idx].axis("off")

plt.suptitle("CVQA Images (English / Teso / Swahili)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Pangea-7B chat template (Qwen2-style special tokens, as documented on the model card)
PANGEA_SYSTEM_MESSAGE = "You are a helpful assistant."

def format_pangea_prompt(user_text):
    """
    Wrap a raw instruction/question string in Pangea-7B's expected chat format,
    including the <image> placeholder token.
    """
    return (
        f"<|im_start|>system\n{PANGEA_SYSTEM_MESSAGE}<|im_end|>\n"
        f"<|im_start|>user\n<image>\n{user_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def build_mcqa_prompt(question, choices):
    """
    Format question + choices into a prompt.
    Instructs the model to answer with only a letter A/B/C/D.
    """
    prompt = (
        "Look at the image carefully and answer the following "
        "multiple choice question by selecting only the letter "
        "A, B, C, or D.\n\n"
        f"Question: {question}\n"
    )
    for label, choice in zip(["A", "B", "C", "D"], choices):
        prompt += f"{label}. {choice}\n"
    prompt += "\nAnswer with only the letter (A, B, C, or D):"
    return prompt

def extract_predicted_label(predicted_text, choices):
    """
    Extract the predicted label (A/B/C/D) from model output.

    Strategy 1: look for A/B/C/D letter in model output.
    Strategy 2: if no letter found, check if model output
                matches any choice text directly.
    If no label is found, defaults to a random choice (A,B,C,D) to ensure a prediction is always made.
    """
    # Strategy 1 — look for letter
    for label in ["A", "B", "C", "D"]:
        if label in predicted_text.upper():
            return label

    # Strategy 2 — match answer text
    for label, choice in zip(["A", "B", "C", "D"], choices):
        if choice.lower() in predicted_text.lower():
            return label

    # Default to a random choice if no label is found
    import random
    return random.choice(["A", "B", "C", "D"])

### Extend Evaluation to Teso Questions

Now, we'll extend the evaluation to also process questions in Teso. This requires specifying the column names in your Google Sheet that contain the Teso questions and their corresponding choices. The `evaluate_sea_mcqa` function will be updated to evaluate both English and Teso questions for each sample.

Pangea-7B was trained on 39 languages, which includes Swahili but not Teso. We still run Teso
questions through the model as a cross-lingual generalisation test — accuracy on Teso should
be interpreted with that in mind.

First, define the Teso-related column names here:

In [ ]:
# Define column names for Teso questions and answers
TESO_QUESTION_COLUMN = "native_question"
TESO_CORRECT_ANSWER_COLUMN = "correct_native"
TESO_WRONG_ANSWER_COLUMNS = ["wrong_native_o1", "wrong_native_o2", "wrong_native_o3"]

print(f"Teso Question Column: {TESO_QUESTION_COLUMN}")
print(f"Teso Correct Answer Column: {TESO_CORRECT_ANSWER_COLUMN}")
print(f"Teso Wrong Answer Columns: {TESO_WRONG_ANSWER_COLUMNS}")

print("\n")

# Define column names for Swahili questions and answers
SWA_QUESTION_COLUMN = "swa_question"
SWA_CORRECT_ANSWER_COLUMN = "correct_swa"
SWA_WRONG_ANSWER_COLUMNS = ["wrong_swa_o1", "wrong_swa_o2", "wrong_swa_o3"]

print(f"Swahili Question Column: {SWA_QUESTION_COLUMN}")
print(f"Swahili Correct Answer Column: {SWA_CORRECT_ANSWER_COLUMN}")
print(f"Swahili Wrong Answer Columns: {SWA_WRONG_ANSWER_COLUMNS}")

Teso Question Column: native_question
Teso Correct Answer Column: correct_native
Teso Wrong Answer Columns: ['wrong_native_o1', 'wrong_native_o2', 'wrong_native_o3']


Swahili Question Column: swa_question
Swahili Correct Answer Column: correct_swa
Swahili Wrong Answer Columns: ['wrong_swa_o1', 'wrong_swa_o2', 'wrong_swa_o3']


Now, the `evaluate_sea_mcqa` function will be modified to process both English and Teso questions for each sample. The results will include a `language` field to distinguish between them.

In [ ]:
# ============================================================
# STEP 3: Run Evaluation (Updated for Trilingual, Pangea-7B)
# ============================================================
def evaluate_sea_mcqa(samples, processor, model, teso_question_col, teso_correct_ans_col, teso_wrong_ans_cols, swa_question_col, swa_correct_ans_col, swa_wrong_ans_cols):
    """
    Evaluate model on SEA-MCQA for both English and Teso questions.
    Accuracy = % of samples where model selects the correct option.
    Random chance baseline = 25% (4 choices).
    """
    predictions = []

    def _evaluate_single_question_language(sample, question_col, correct_ans_col, wrong_ans_cols, language_tag):
        question    = sample[question_col]
        correct_ans = sample[correct_ans_col]
        wrong_1     = sample[wrong_ans_cols[0]]
        wrong_2     = sample[wrong_ans_cols[1]]
        wrong_3     = sample[wrong_ans_cols[2]]

        # Shuffle choices so correct answer isn't always in same position
        choices = [correct_ans, wrong_1, wrong_2, wrong_3]
        random.shuffle(choices)
        correct_label = ["A", "B", "C", "D"][choices.index(correct_ans)]

        # Build prompt (raw instruction text, then wrap in Pangea's chat template)
        mcqa_prompt = build_mcqa_prompt(question, choices)
        chat_prompt = format_pangea_prompt(mcqa_prompt)

        predicted_text = "ERROR"
        predicted_label = None
        is_correct = False

        try:
            # Load image from dataset
            image = sample["image"].convert("RGB")

            # Run model
            inputs = processor(images=image, text=chat_prompt, return_tensors="pt").to(model.device, torch.float16)

            with torch.inference_mode():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=20,
                    do_sample=False,
                )

            # Only decode the newly generated tokens (Pangea/LLaVA-NeXT returns the full
            # sequence including the prompt, unlike BLIP's encoder-decoder `generate`).
            generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
            predicted_text = processor.decode(
                generated_ids, skip_special_tokens=True
            ).strip()

            # Extract predicted label using both strategies
            predicted_label = extract_predicted_label(predicted_text, choices)

            # Check correctness
            is_correct = (predicted_label == correct_label)

        except Exception as e:
            print(f"Error processing sample ID {sample['ID']} ({language_tag} question): {e}")

        return {
            "sample_id":       sample["ID"],
            "category":        sample["Category"],
            "language":        language_tag,
            "question":        question,
            "choices":         {l: c for l, c in zip(["A","B","C","D"], choices)},
            "correct_answer":  correct_ans,
            "correct_label":   correct_label,
            "predicted_text":  predicted_text,
            "predicted_label": predicted_label,
            "is_correct":      is_correct
        }

    for i, sample in enumerate(samples):
        # Evaluate English question
        english_prediction = _evaluate_single_question_language(
            sample, "eng_question", "correct_en",
            ["wrong_en_o1", "wrong_en_o2", "wrong_en_o3"], "English"
        )
        predictions.append(english_prediction)

        # Evaluate Teso question
        teso_prediction = _evaluate_single_question_language(
            sample, teso_question_col, teso_correct_ans_col,
            teso_wrong_ans_cols, "Teso"
        )
        predictions.append(teso_prediction)


        # Evaluate Swahili question
        swa_prediction = _evaluate_single_question_language(
            sample, swa_question_col, swa_correct_ans_col,
            swa_wrong_ans_cols, "Swahili"
        )
        predictions.append(swa_prediction)


    overall_correct = sum(p['is_correct'] for p in predictions)
    overall_accuracy = (overall_correct / len(predictions)) * 100 if predictions else 0

    return overall_accuracy, predictions

# Call the updated evaluation function
accuracy, detailed_predictions = evaluate_sea_mcqa(
    samples, processor, model,
    TESO_QUESTION_COLUMN, TESO_CORRECT_ANSWER_COLUMN, TESO_WRONG_ANSWER_COLUMNS,
    SWA_QUESTION_COLUMN, SWA_CORRECT_ANSWER_COLUMN, SWA_WRONG_ANSWER_COLUMNS
)

print("Evaluation complete for English, Teso, and Swahili questions.")

[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
[transformers] Setting `pad_token_id` to `eo

Evaluation complete for English, Teso, and Swahili questions.


###  Print Results (Updated for Trilingual)

This section now displays the detailed evaluation results for English, Teso, and Swahili questions.

In [ ]:
# ============================================================
# STEP 4: Print Results (Updated for Trilingual)
# ============================================================
print("\n" + "="*60)
print("          MCQA Evaluation Results (English, Teso, and Swahili)")
print("="*60)

correct_english = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'English')
total_english = sum(1 for p in detailed_predictions if p['language'] == 'English')
accuracy_english = (correct_english / total_english) * 100 if total_english else 0

correct_teso = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'Teso')
total_teso = sum(1 for p in detailed_predictions if p['language'] == 'Teso')
accuracy_teso = (correct_teso / total_teso) * 100 if total_teso else 0


correct_swa = sum(p['is_correct'] for p in detailed_predictions if p['language'] == 'Swahili')
total_swa = sum(1 for p in detailed_predictions if p['language'] == 'Swahili')
accuracy_swa = (correct_swa / total_swa) * 100 if total_teso else 0


for i, pred in enumerate(detailed_predictions, 1):
    status = "✓" if pred["is_correct"] else "✗"
    print(f"\n[{status}] Q{pred['sample_id']} ({pred['language']}) [{pred['category']}]")
    print(f"  Question: {pred['question']}")
    for label, choice in pred["choices"].items():
        marker = " ← correct" if choice == pred["correct_answer"] else ""
        print(f"  {label}. {choice}{marker}")
    print(f"  Model output:    {pred['predicted_text']}")
    print(f"  Predicted label: {pred['predicted_label']}  "
          f"|  Correct label: {pred['correct_label']}")

print("\n" + "="*60)
print(f"Overall Accuracy:        {accuracy:.2f}% ({sum(p['is_correct'] for p in detailed_predictions)}/{len(detailed_predictions)}) ")
print(f"English Accuracy:        {accuracy_english:.2f}% ({correct_english}/{total_english})")
print(f"Teso Accuracy:           {accuracy_teso:.2f}% ({correct_teso}/{total_teso})")
print(f"Swahili Accuracy:        {accuracy_swa:.2f}% ({correct_swa}/{total_swa})")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*60)


          MCQA Evaluation Results (English, Teso, and Swahili)

[✓] Q1 (English) [Image 1 Questions]
  Question: Which of the baskets are pictured, what are they called in this country, and what is their purpose?
  A. Uteo. Crafted from reeds or palm leaves. It is traditionally found in households and was used to separate the chaff from grain.Its commonly used to hold cereals, in market places, and in homes. ← correct
  B. Mesob baskets. Individual strands of grass or palm are dyed with vibrant colors and utilized for communal serving holding traditional meals
  C. Bolga baskets. Made from thick, dried elephant grass. They are also very durable because of the tight weaving and are used as market or beach bags, home decor, and for storage and organization.
  D. Emotokaa. Kanueraitere nakapoloni itomate adakia inyamata luipwaka katoni luitijimete inyamene ayari namakegwelete
  Model output:    The baskets pictured in the image are most likely Bolga baskets. These are made from thick, dr

### Visualise Results (Updated for Trilingual)

This section now visualises the evaluation results, indicating correctness for English, Teso, and Swahili questions.

In [ ]:
# ============================================================
# STEP 5: Visualise Results (Updated for Trilingual)
# ============================================================

# Create a figure with a grid of subplots for each sample, showing English, Teso, and
# Swahili results. Each image appears once, with three result rows beneath it.

num_samples = len(samples)
fig, axes = plt.subplots(2, num_samples, figsize=(4 * num_samples, 8)) # row 0: image, row 1: results text

if num_samples == 1:
    axes = axes.reshape(2, 1)

for i, sample in enumerate(samples):
    # Find predictions for this sample ID
    sample_preds = [p for p in detailed_predictions if p['sample_id'] == sample['ID']]

    english_pred = next((p for p in sample_preds if p['language'] == 'English'), None)
    teso_pred = next((p for p in sample_preds if p['language'] == 'Teso'), None)
    swa_pred = next((p for p in sample_preds if p['language'] == 'Swahili'), None)

    # Display image in the top row
    axes[0, i].imshow(sample["image"])
    axes[0, i].set_title(f"Q{sample['ID']} [{sample['Category']}]", fontsize=9)
    axes[0, i].axis("off")

    # Display English results
    if english_pred:
        color_en  = "green" if english_pred["is_correct"] else "red"
        status_en = "✓" if english_pred["is_correct"] else "✗"
        axes[1, i].text(0.5, 0.83,
                        f"English: {status_en} \nPred: {english_pred['predicted_label']} | Correct: {english_pred['correct_label']}",
                        fontsize=8, color=color_en, ha='center', va='center',
                        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    else:
        axes[1, i].text(0.5, 0.83, "English: No Data", fontsize=8, color='gray', ha='center', va='center')

    # Display Teso results
    if teso_pred:
        color_te  = "green" if teso_pred["is_correct"] else "red"
        status_te = "✓" if teso_pred["is_correct"] else "✗"
        axes[1, i].text(0.5, 0.5,
                        f"Teso: {status_te} \nPred: {teso_pred['predicted_label']} | Correct: {teso_pred['correct_label']}",
                        fontsize=8, color=color_te, ha='center', va='center',
                        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    else:
        axes[1, i].text(0.5, 0.5, "Teso: No Data", fontsize=8, color='gray', ha='center', va='center')

    # Display Swahili results
    if swa_pred:
        color_sw  = "green" if swa_pred["is_correct"] else "red"
        status_sw = "✓" if swa_pred["is_correct"] else "✗"
        axes[1, i].text(0.5, 0.17,
                        f"Swahili: {status_sw} \nPred: {swa_pred['predicted_label']} | Correct: {swa_pred['correct_label']}",
                        fontsize=8, color=color_sw, ha='center', va='center',
                        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))
    else:
        axes[1, i].text(0.5, 0.17, "Swahili: No Data", fontsize=8, color='gray', ha='center', va='center')

    axes[1, i].axis('off')

plt.suptitle(
    f"SEA-MCQA Results (English / Teso / Swahili) — Pangea-7B — Overall Accuracy: {accuracy:.1f}% "
    f"(Random baseline: 25%)",
    fontsize=12
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make room for suptitle
plt.show()


print("\n" + "="*55)
print("       MCQA Evaluation Results (English/Teso/Swahili)")
print("="*55)

print(f"\nOverall Accuracy:        {accuracy:.2f}%")
print(f"Correct:                 "
      f"{sum(p['is_correct'] for p in detailed_predictions)}/{len(detailed_predictions)}")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*55)